# 01c - Análise Espectral: Resíduo Estatístico do StyleGAN

Imagens geradas por GANs como o StyleGAN deixam assinaturas no domínio de frequência que CNNs aprendem trivialmente. Este notebook visualiza esses resíduos via FFT e mostra como a compressão JPEG os remove.

Comparamos o espectro médio de:
- Imagens **reais** (test split do dataset 140k Real and Fake Faces)
- Imagens **falsas originais** (StyleGAN, sem compressão)
- Imagens **falsas comprimidas** (StyleGAN + JPEG q=50)

In [ ]:
import io
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm import tqdm

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent
_data_root_file = PROJECT_ROOT / "data_root.env"
DATA_ROOT    = Path(_data_root_file.read_text().strip()) if _data_root_file.exists() else PROJECT_ROOT / "data"

RAW_DIR     = DATA_ROOT / "raw" / "140k_faces" / "real_vs_fake" / "real-vs-fake"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

N_SAMPLES  = 5000   # test split completo
SEED       = 42
IMAGE_SIZE = 128

print("Imagens:", RAW_DIR)
print("Amostras por grupo:", N_SAMPLES)
print("Compressão feita em memória — nenhuma imagem salva em disco.")

## 1. Funções

In [ ]:
def load_images(folder, n, seed=42, size=128):
    paths = sorted(folder.glob("*.jpg"))
    paths = random.Random(seed).sample(paths, min(n, len(paths)))
    imgs = []
    for p in tqdm(paths, desc=f"Carregando {folder.parent.name}/{folder.name}"):
        img = Image.open(p).convert("L").resize((size, size))
        imgs.append(np.array(img, dtype=np.float32) / 255.0)
    return np.stack(imgs), paths

def compress_in_memory(raw_paths, quality, size=128):
    imgs = []
    for p in raw_paths:
        img = Image.open(p).convert("RGB").resize((size, size))
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=quality)
        buf.seek(0)
        img_c = Image.open(buf).copy().convert("L")
        imgs.append(np.array(img_c, dtype=np.float32) / 255.0)
    return np.stack(imgs)

def mean_fft_magnitude(images):
    spectra = []
    for img in images:
        f = np.fft.fft2(img)
        f_shift = np.fft.fftshift(f)
        magnitude = np.log1p(np.abs(f_shift))
        spectra.append(magnitude)
    return np.mean(spectra, axis=0)

def radial_profile(spectrum):
    center = np.array(spectrum.shape) // 2
    y, x = np.indices(spectrum.shape)
    r = np.sqrt((x - center[1])**2 + (y - center[0])**2).astype(int)
    r_max = min(center)
    profile = np.array([spectrum[r == i].mean() for i in range(r_max)])
    return profile

print("Funções definidas.")

## 2. Carregamento das Imagens

In [ ]:
print("Carregando imagens reais...")
real_imgs, real_paths = load_images(RAW_DIR / "test" / "real", N_SAMPLES, SEED, IMAGE_SIZE)

print("Carregando imagens falsas originais...")
fake_imgs, fake_paths = load_images(RAW_DIR / "test" / "fake", N_SAMPLES, SEED, IMAGE_SIZE)

print(f"Carregadas: {len(real_imgs)} reais | {len(fake_imgs)} falsas")

## 3. Varredura de Qualidade JPEG (100 → 50, de 5 em 5)

Comprime as imagens falsas em memória para cada nível de qualidade e plota o perfil radial. Faixa testada: q=100 a q=50, onde se espera encontrar o mínimo de distância espectral.

In [ ]:
QUALITY_RANGE = range(100, 45, -5)  # 100, 95, 90, ..., 50
cmap_sweep = plt.cm.plasma

real_profile = radial_profile(mean_fft_magnitude(real_imgs))
fake_profile = radial_profile(mean_fft_magnitude(fake_imgs))
freqs = np.arange(len(real_profile))

profiles_by_quality = {}
for q in tqdm(QUALITY_RANGE, desc="Varredura de qualidade JPEG"):
    imgs_q = compress_in_memory(fake_paths, quality=q, size=IMAGE_SIZE)
    profiles_by_quality[q] = radial_profile(mean_fft_magnitude(imgs_q))

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(freqs, real_profile, color="steelblue", linewidth=2.5, label="Real (140k)", zorder=10)
ax.plot(freqs, fake_profile, color="black", linewidth=2, linestyle="--", label="Fake — sem compressão", zorder=9)

qualities = list(profiles_by_quality.keys())
colors = cmap_sweep(np.linspace(0, 1, len(qualities)))

for (q, profile), color in zip(profiles_by_quality.items(), colors):
    ax.plot(freqs, profile, color=color, alpha=0.7, linewidth=0.9)

sm = plt.cm.ScalarMappable(cmap=cmap_sweep, norm=plt.Normalize(vmin=min(qualities), vmax=max(qualities)))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label("Qualidade JPEG")

ax.set_xlabel("Frequência radial (pixels)")
ax.set_ylabel("Magnitude média (log)")
ax.set_title("Perfil Radial por Nível de Compressão JPEG (Fake — StyleGAN)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "espectro_varredura_qualidade.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Interpretação Automática — Qualidade Ideal

Calcula a distância (MSE) entre o perfil espectral de cada nível de compressão e o perfil real. Como a curva MSE tem formato de **U** (diminui e depois aumenta), o critério correto é o **mínimo global** — ponto onde o espectro fake é mais próximo do real.

In [ ]:
qualities_sorted = sorted(profiles_by_quality.keys(), reverse=True)  # 100 -> 5
mse_values = [
    float(np.mean((profiles_by_quality[q] - real_profile) ** 2))
    for q in qualities_sorted
]
mse_no_compression = float(np.mean((fake_profile - real_profile) ** 2))

# mínimo global — qualidade onde o espectro fake é mais próximo do real
best_idx     = int(np.argmin(mse_values))
best_quality = qualities_sorted[best_idx]
best_mse     = mse_values[best_idx]
mse_reduction = (1 - best_mse / mse_no_compression) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(qualities_sorted, mse_values, marker="o", markersize=4, color="tomato", label="MSE fake vs real")
ax.axhline(mse_no_compression, color="black", linestyle="--", alpha=0.6, label="MSE sem compressão")
ax.axvline(best_quality, color="steelblue", linestyle="--", linewidth=1.5, label=f"Mínimo: q={best_quality}")
ax.scatter([best_quality], [best_mse], color="steelblue", s=80, zorder=5)
ax.invert_xaxis()
ax.set_xlabel("Qualidade JPEG")
ax.set_ylabel("MSE (perfil fake − real)")
ax.set_title("Distância Espectral por Nível de Compressão")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "espectro_mse_vs_qualidade.png", dpi=150, bbox_inches="tight")
plt.show()

print("=" * 60)
print(f"MSE sem compressão:              {mse_no_compression:.6f}")
print(f"MSE mínimo (q={best_quality:>3}):          {best_mse:.6f}")
print(f"Redução de MSE no mínimo:        {mse_reduction:.1f}%")
print("=" * 60)
print(f"\n→ Qualidade de menor distância espectral: JPEG q={best_quality}")
print(f"\nNota: a curva tem formato U — compressão abaixo de q={best_quality}")
print(f"  introduz artefatos JPEG que afastam o espectro fake do real,")
print(f"  não remove artefatos do StyleGAN.")